In [1]:
import random
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

#TODO: Do I need to implement SEED here? Am I implementing SEED correctly by passing it to the trainer and that is it?
'''#SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True  # same seed at GPU as well
    torch.backends.cudnn.benchmark = False  # same seed at GPU as well'''


Using device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU


"#SEED = 42\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\nif DEVICE.type == 'cuda':\n    torch.cuda.manual_seed_all(SEED)\n    torch.backends.cudnn.deterministic = True  # same seed at GPU as well\n    torch.backends.cudnn.benchmark = False  # same seed at GPU as well"

In [2]:
from BioRED_python_download.biored import BioredDataset
from transformers import DistilBertModel, DistilBertTokenizer, DistilBertForSequenceClassification, DataCollatorWithPadding
from nltk.tokenize import word_tokenize
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DistilBertModel, DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import AutoTokenizer
import numpy as np
from transformers import AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification
import itertools


# Select existing config
builder = BioredDataset(config_name="biored_bigbio_kb")

# Download and prepare the dataset
builder.download_and_prepare()

# Load dataset splits. That is, NO NEED FOR SEED (IN DATA SPLIT, elsewhere SEED is needed) because we are using pre-defined splits from BioRED
dataset = builder.as_dataset()

train_data = dataset["train"]
test_data = dataset["test"]
val_data = dataset["validation"]

'''#TODO: Remember to change DEBUG to False if running experiments.
DEBUG = True
if DEBUG:
    train_data = train_data.select(range(32))
    val_data = val_data.select(range(16))
    test_data = test_data.select(range(8))
    print("train_set length DEBUGGING:", len(train_data), "val_set length DEBUGGING:", len(val_data), "test_set length DEBUGGING:", len(test_data))'''

print("Column names:\n", train_data.column_names)
print("Features:\n", train_data.features)
print("Entities:\n", train_data.features["entities"])
print("Number of columns:", train_data.num_columns)

print("First data sample in train:\n", train_data[0])

sample = train_data[0]
print(f"Title/Abstract: {sample['passages'][0]['text']}")
print(f"First Entity: {sample['entities'][0]}")
print(f"First Relation: {sample['relations'][0]}")

'''print("First data sample in train (line by line):")
print(train_data[0]["id"])
print(train_data[0]["document_id"])
print(train_data[0]["passages"])
print(train_data[0]["entities"])
print(train_data[0]["events"])
print(train_data[0]["coreferences"])
print(train_data[0]["relations"])'''

print("Size of training subset (i.e. number of data samples):", train_data.num_rows)
print("Size of validation subset (i.e. number of data samples):", val_data.num_rows)
print("Size of testing subset (i.e. number of data samples):", test_data.num_rows)


Column names:
 ['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations']
Features:
 {'id': Value('string'), 'document_id': Value('string'), 'passages': List({'id': Value('string'), 'type': Value('string'), 'text': List(Value('string')), 'offsets': List(List(Value('int32')))}), 'entities': List({'id': Value('string'), 'type': Value('string'), 'text': List(Value('string')), 'offsets': List(List(Value('int32'))), 'normalized': List({'db_name': Value('string'), 'db_id': Value('string')})}), 'events': List({'id': Value('string'), 'type': Value('string'), 'trigger': {'text': List(Value('string')), 'offsets': List(List(Value('int32')))}, 'arguments': List({'role': Value('string'), 'ref_id': Value('string')})}), 'coreferences': List({'id': Value('string'), 'entity_ids': List(Value('string'))}), 'relations': List({'id': Value('string'), 'type': Value('string'), 'arg1_id': Value('string'), 'arg2_id': Value('string'), 'normalized': List({'db_name': Value('string'), 'db_

In [3]:
from datasets import Dataset

# 1. Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained('google-bert/bert-base-uncased', add_prefix_space=True)
#tokenizer = BertTokenizer.from_pretrained('google-bert/bert-base-uncased', add_prefix_space=True)

# Map BioRED entity types to ID integers. This code was inspired by https://huggingface.co/docs/transformers/tasks/token_classification, the huggingFace tutorial on how to perform NER with BERT
label_list = [ "O",
               "B-ChemicalEntity", "I-ChemicalEntity",
               "B-DiseaseOrPhenotypicFeature","I-DiseaseOrPhenotypicFeature",
               "B-GeneOrGeneProduct","I-GeneOrGeneProduct",
               "B-SequenceVariant","I-SequenceVariant",
               "B-OrganismTaxon","I-OrganismTaxon",
               "B-CellLine", "I-CellLine"
               ]#["O", "B-Gene", "I-Gene", "B-Disease", "I-Disease", "B-Chemical", "I-Chemical"]  # Simplified for example. #["GeneOrGeneProduct", "DiseaseOrPhenotypicFeature", "ChemicalEntity", "SequenceVariant", "OrganismTaxon", "CellLine"]
label_to_id = {l: i for i, l in enumerate(label_list)}

# Map BioRED relation types to ID integers
relation_list = [
    "None", #not in original BioRED dataset, but added for logic to determine label
    "Association",
    "Bind",
    "Comparison",
    "Conversion",
    "Cotreatment",
    "Drug_Interaction",
    "Negative_Correlation",
    "Positive_Correlation"
]
relation_to_id = {r: i for i, r in enumerate(relation_list)}
id_to_relation = {i: r for i, r in enumerate(relation_list)}

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer) # only needed for token-level classification tasks, NOT sequence-level classification task. That is, the data_collator is needed for NER, but NOT for RE. The default DataCollatorWithPadding called by Trainer is all RE needs for padding )

def preprocess_biored_for_ner(examples):
    # BioRED KB has 'passages' containing the full text
    texts = [
        " ".join(
            passage_text
            for passage in p
            for passage_text in passage["text"]
        )
        for p in examples["passages"]
    ]

    tokenized_inputs = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_offsets_mapping=True  # CRITICAL for alignment
    )

    labels = []
    for i, document_entities in enumerate(examples["entities"]):
        offset_mapping = tokenized_inputs["offset_mapping"][i]
        doc_labels = np.zeros(len(offset_mapping), dtype=int)

        for entity in document_entities:
            start_char = entity["offsets"][0][0]
            end_char = entity["offsets"][0][1]
            entity_type = entity["type"]
            if entity_type not in [
                "ChemicalEntity",
                "DiseaseOrPhenotypicFeature",
                "GeneOrGeneProduct",
                "SequenceVariant",
                "OrganismTaxon",
                "CellLine"
            ]:
                print(entity_type + " " + str(start_char) + " " + str(end_char))

            # Map tokens to labels based on character offsets
            inside = False
            for idx, (start, end) in enumerate(offset_mapping):
                if start is None or start == end: #if start == end:  # Skip special tokens like [CLS] or [PAD]
                    doc_labels[idx] = -100  # Standard ignore index for loss
                    continue

                if not (end <= start_char or start >= end_char): #if start >= start_char and end <= end_char
                    # Logic for B- and I- tags
                    if not inside:
                        prefix = "B-"
                        inside = True
                    else:
                        prefix = "I-"
                    label_str = f"{prefix}{entity_type}"

                    if label_str in label_to_id:
                        doc_labels[idx] = label_to_id[label_str]
                    else:
                        raise ValueError(f"Unknown label: {label_str}")
        labels.append(doc_labels.tolist())

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


def preprocess_biored_for_re(example):
    entities = example["entities"]
    existing_relations = {(r["arg1_id"], r["arg2_id"]): r["type"] for r in example["relations"]}

    for r in example["relations"]:
        #print(r['type'])
        if r['type'] not in relation_list:
            print("RELATION LIST TEST, relation missing:\n", r, "\n ", r['type'])

    #Alternative to be tried out if for training current implementation of method returning a dictionary in the way it currently does (even though as per HuggingFace it does need to return a dictionary) causes problems
    texts = []
    labels = []

    context = " ".join(example["passages"][0]["text"]) #context = " ".join(example["passages"][0]["text"])  # Simple abstract-level context
    # Generate every possible combination of 2 entities in the document
    for ent1, ent2 in itertools.combinations(entities, 2):
        pair = (ent1["id"], ent2["id"])
        reverse_pair = (ent2["id"], ent1["id"])

        # Determine label (Positive relation or 'None' for Negative Sampling)
        relation_type = "None"
        if pair in existing_relations:
            relation_type = existing_relations[pair]
        elif reverse_pair in existing_relations:
            relation_type = existing_relations[reverse_pair]

        # Create a text representation with Entity Markers and feed entity pair as a sentence pair to the tokenizer
        # "Gene [E1] Factor-6 [/E1] is linked to [E2] Diabetes [/E2]"
        entity_pair_str = f"{ent1['text'][0]} [SEP] {ent2['text'][0]}"

        texts.append((context, entity_pair_str))  # pair-level tokenization, unlike for NER, which requires document-level tokenization and hence, why input is tokenized at a different stage / before preparing the data for the NER module
        if relation_type not in relation_to_id:
            raise ValueError(f"Unknown relation type: {relation_type}")
        labels.append(relation_to_id[relation_type])  # e.g. 1

        # Tokenize all pairs for this document at once
    tokenized = tokenizer(
        [t[0] for t in texts],   # contexts
        [t[1] for t in texts],   # entity pairs
        truncation=True,
        padding="max_length",
        max_length=512
    )

    tokenized["labels"] = labels
    return tokenized








def flatten_re_dataset(re_mapped):
    """
    preprocess_biored_for_re returns variable-length lists per document
    (one entry per entity pair) such that .map function can be applied to preprocess_biored_for_re, but this is incompatible with Trainer, which expexts a different format. This flattens them into individual samples.
    """
    all_input_ids      = []
    all_attention_mask = []
    all_token_type_ids = []
    all_labels         = []

    for sample in re_mapped:
        ids = sample["input_ids"]
        # Each document produces a list of lists (one per entity pair)
        if isinstance(ids[0], list):
            all_input_ids.extend(ids)
            all_attention_mask.extend(sample["attention_mask"])
            if "token_type_ids" in sample:
                all_token_type_ids.extend(sample["token_type_ids"])
            all_labels.extend(sample["labels"])
        else:
            all_input_ids.append(ids)
            all_attention_mask.append(sample["attention_mask"])
            if "token_type_ids" in sample:
                all_token_type_ids.append(sample["token_type_ids"])
            all_labels.append(sample["labels"])

    flat = {
        "input_ids":      all_input_ids,
        "attention_mask": all_attention_mask,
        "labels":         all_labels
    }
    if all_token_type_ids:
        flat["token_type_ids"] = all_token_type_ids

    return Dataset.from_dict(flat)


In [4]:
if __name__ == "__main__":
    train_ner_data = train_data.map(preprocess_biored_for_ner, batched=True) # apply to train subset
    #TODO: what is examples and doesnt this method take it as input? If so, why is it called without any arguments?
    val_ner_data = val_data.map(preprocess_biored_for_ner, batched=True)  # apply to val subset
    test_ner_data = test_data.map(preprocess_biored_for_ner, batched=True)  # apply to test subset

    #print(train_ner_data[0])
    #print("NER data preprocessing complete")

    train_re_data = train_data.map(preprocess_biored_for_re) #, batched=True) # apply to train subset
    val_re_data = val_data.map(preprocess_biored_for_re) #, batched=True) # apply to val subset
    test_re_data = test_data.map(preprocess_biored_for_re) #, batched=True)  # apply to test subset

    #print(train_re_data[0])
    #print("RE data preprocessing complete")

    train_re_data = flatten_re_dataset(train_re_data)
    val_re_data   = flatten_re_dataset(val_re_data)
    test_re_data  = flatten_re_dataset(test_re_data)

    train_ner_data.set_format(type="torch")#, output_all_columns=True) # convert it to tensor for torch and Transformers to be able to handle the training subset for NER
    val_ner_data.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the val subset for NER
    test_ner_data.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the test subset for NER

    train_re_data.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the training subset for RE
    val_re_data.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the val subset for RE
    test_re_data.set_format(type="torch") # convert it to tensor for torch and Transformers to be able to handle the test subset for RE

    print(train_ner_data.column_names)
    print(train_re_data.column_names)

    # Remove offset_mapping after preprocessing — model doesn't need it
    train_ner_data = train_ner_data.remove_columns(["offset_mapping"])
    val_ner_data = val_ner_data.remove_columns(["offset_mapping"])
    test_ner_data = test_ner_data.remove_columns(["offset_mapping"])

    print(train_ner_data.column_names)
    print(train_re_data.column_names)

['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations', 'input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'labels']
['input_ids', 'attention_mask', 'labels', 'token_type_ids']
['id', 'document_id', 'passages', 'entities', 'events', 'coreferences', 'relations', 'input_ids', 'token_type_ids', 'attention_mask', 'labels']
['input_ids', 'attention_mask', 'labels', 'token_type_ids']


In [5]:
# use BERT for NER
import random
import torch
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
import evaluate
seqeval = evaluate.load("seqeval")

ner_model = AutoModelForTokenClassification.from_pretrained( #no explicit word embeddings are created because these are automatically created and used internally by BERT
    "google-bert/bert-base-uncased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id=label_to_id
)

#https://huggingface.co/docs/transformers/v5.8.0/en/main_classes/trainer#transformers.TrainingArguments
def train_ner(train_ner_data_arg, val_ner_data_arg):
    ner_training_args = TrainingArguments(
        output_dir="./ner_results",
        per_device_train_batch_size= 16, #train_config_ner['batch_size'],
        per_device_eval_batch_size= 16, #train_config_ner['eval_batch_size'],
        num_train_epochs= 3, #train_config_ner['num_epochs'],
        #max_steps=25, # as max_steps rule of thumb is number of samples (400 in train) divided by per_device_train_batch_size(16)xnum_gpus#defult value for now,
        eval_strategy="epoch",
        save_strategy="best", #train_config_ner['save_strategy'],
        learning_rate=2e-5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",#train_config_ner['metric_for_best_model'],
        seed=42 # DataCollatorForLanguageModeling includes seed, but DataCollatorForTokenClassification does NOT. So, we need to set it here.
    )

    ner_trainer = Trainer(
        model=ner_model,
        args=ner_training_args,
        train_dataset=train_ner_data_arg,
        eval_dataset=val_ner_data_arg,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_ner
    )
    ner_trainer.train()
    return ner_trainer



def compute_metrics_ner(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly in

In [6]:
# use BERT for RE
re_model = AutoModelForSequenceClassification.from_pretrained(#no explicit word embeddings are created because these are automatically created and used internally by BERT
    "google-bert/bert-base-uncased",
    num_labels=len(relation_list),
    id2label=id_to_relation,
    label2id=relation_to_id
)

#print(re_model)

'''re_model = DistilBertForSequenceClassification.from_pretrained(#no explicit word embeddings are created because these are automatically created and used internally by BERT
    "google-bert/bert-base-uncased",
    num_labels=len(relation_list),
    id2label=id_to_relation,
    label2id=relation_to_id
)'''

#https://huggingface.co/docs/transformers/v5.8.0/en/main_classes/trainer#transformers.TrainingArguments: "load_best_model_at_end (bool, optional, defaults to False) — Load the best checkpoint at the end of training. Requires eval_strategy to be set. When enabled, the best checkpoint is always saved (see save_total_limit). When `True`, `save_strategy` must match `eval_strategy` (unless `save_strategy` is `"best"`), and if using `"steps"`, `save_steps` must be a multiple of `eval_steps`."

def train_re(train_re_data_arg, val_re_data_arg):
    #print(train_re_data[0])
    re_training_args = TrainingArguments(
        output_dir="./re_results",
        per_device_train_batch_size= 16, #train_config_re['batch_size'],
        per_device_eval_batch_size= 16, #train_config_re['eval_batch_size'],
        num_train_epochs=3,
        max_steps=100, #100 for degugging #25, # as max_steps rule of thumb is number of samples (400 in train) divided by per_device_train_batch_size(16)xnum_gpus #defult value for now, 25 # TODO: Remove max_steps entirely WHEN RUNNING EXPERIMENTS
        eval_strategy="epoch",
        save_strategy="best", #train_config_re['save_strategy'],
        learning_rate=2e-5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",#train_config_re['metric_for_best_model'],
        seed=42 # DataCollatorForLanguageModeling includes seed, but DataCollatorForTokenClassification does NOT. So, we need to set it here.
    )

    re_trainer = Trainer(
        model=re_model,
        args=re_training_args,
        train_dataset=train_re_data_arg,
        eval_dataset=val_re_data_arg,
        processing_class=tokenizer,
    )

    re_trainer.train()
    return re_trainer

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
if __name__ == "__main__":

    ner_trainer = train_ner(train_ner_data, val_ner_data)
    re_trainer  = train_re(train_re_data,  val_re_data)

    ner_trainer.save_model("./ner_model_final")
    re_trainer.save_model("./re_model_final")

    print("Pipeline 1 training complete.")

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.750565,0.180240,0.197831,0.188627,0.793862
2,No log,0.503755,0.339094,0.408265,0.370479,0.846819
3,No log,0.437268,0.381993,0.466295,0.419955,0.866859


C:\Users\lamus\anaconda3\envs\NLP_GPU_conda_env\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\lamus\anaconda3\envs\NLP_GPU_conda_env\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\lamus\anaconda3\envs\NLP_GPU_conda_env\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Pipeline 1 training complete.
